# Credit Risk Project

## Feature Engineering

### Author: Aaron Thomas

<b>Goal</b>: The goal of this notebook is to perform feature engineering on the credit risk dataset. This includes creating new features, transforming existing features, and preparing the data for modeling.

## Table of Contents

<ol>

<li>Introduction</li>
<li>Import Libraries</li>
<li>Data Cleaning</li>
<li>Feature Engineering</li>
<li>Feature Transformation</li>
<li>Save Processed Data</li>
<li>Conclusion</li>
</ol>

## 1. Introduction

The purpose of this notebook is to perform feature engineering on the credit risk dataset. This involves creating new features, transforming existing features, and preparing the data for modeling. The goal is to enhance the predictive power of the dataset and improve the performance of machine learning models.

## 2. Import Libraries

Before starting the feature engineering process, the following libraries will be imported:

In [1]:
import pandas as pd

## 3. Data Cleaning <a id='data_cleaning'></a>

The purpose of data cleaning is to ensure that the dataset is free from errors, inconsistencies, and missing values.

#### Load data

The following code is used to load the dataset:

In [2]:
df = pd.read_csv('../data/credit_risk_dataset.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


#### Check for missing values

This step checks for any missing values and ensures that the data is complete and ready for analysis.

In [3]:
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Based on the results above, the following observations can be made:
- `person_emp_length` has 895 missing values
- `loan_int_rate` has 3116 missing values

To address these missing values, the following steps are taken:
- For `person_emp_length`, the missing values are filled with the median value of the column.
- Based on the results from EDA, the missing values in `loan_int_rate` are filled with the median interest rate for each loan grade.

For `person_emp_length`, the following code is used to fill the missing values with the median:

In [4]:
df['person_emp_length'].fillna(df['person_emp_length'].median(), inplace=True)

C:\Users\astho\AppData\Local\Temp\ipykernel_35088\3490707621.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['person_emp_length'].fillna(df['person_emp_length'].median(), inplace=True)


For `loan_int_rate`, the following code is used to fill the missing values with the median interest rate for each loan grade:

In [5]:
df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(lambda x: x.fillna(x.median()))

In [6]:
df.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

It can be seen that there are no more missing values in the dataset.

## 4. Feature Engineering <a id='feature_engineering'></a>

The purpose of feature engineering is to create new features that can help improve the performance of the model. In this section, new features will be created based on existing ones. 

#### Data Types

The purpose of checking the data types is to understand the nature of each column and ensure that they are appropriate for modeling. 

In [7]:
df.dtypes

person_age                      int64
person_income                   int64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length      int64
dtype: object

Based on the results above, four features (`person_home_ownership`, `loan_intent`, `loan_grade`, and `cb_person_default_on_file`) are of object type, which indicates they are categorical variables. These features will need to be encoded before they can be used in modeling. The remaining features are of numerical type and can be used directly in modeling after any necessary transformations.

#### Add New Feature

For this section, a new feature called `high_loan_burden` is created, which indicates whether the `loan_percent_income` is greater than 0.5. This feature can help identify borrowers who have a high loan burden relative to their income, which could be an important factor in assessing credit risk. The following code is used to create the new feature:

In [8]:
df['high_loan_burden'] = (df['loan_percent_income'] > 0.5).astype(int)

In addition, a new feature called `age_group` is created by categorizing the `person_age` feature into three groups: 'Young' (age < 30), 'Middle-aged' (age 30-50), and 'Senior' (age > 50). This can help capture any non-linear relationships between age and credit risk. The following code is used to create the new feature:

In [9]:
df['age_group'] = pd.cut(df['person_age'], bins=[0, 30, 50, float('inf')], labels=['Young', 'Middle-aged', 'Senior'])

In the next section, this feature will be encoded to be used in modeling.

## 5. Feature Transformation <a id='feature_transformation'></a>

The purpose of feature transformation is to modify the existing features to improve the performance of the model.  As mentioned in the third section (Feature Engineering), the four categorical features (`person_home_ownership`, `loan_intent`, `loan_grade`, and `cb_person_default_on_file`) will be encoded.

To encode the categorical variables `person_home_ownership`, `age_group`, and `loan_intent`, the following code is used (Note: These two variables have no order):

In [10]:
df = pd.get_dummies(df, columns = ['person_home_ownership', 'age_group', 'loan_intent'], drop_first = True) # drop_first = True to avoid dummy variable trap

To encode the categorical variable `loan_grade`, the following code is used (Note: This variable has an order from A to G):

In [11]:
grade_mapping = {'A': 1, 'B': 2, 'C': 3, 
                 'D': 4, 'E': 5, 'F': 6, 
                 'G': 7}

df['loan_grade'] = df['loan_grade'].map(grade_mapping)

To encode the categorical variable `cb_person_default_on_file`, the following code is used (Note: This variable has an order, where 'Y' indicates a higher risk than 'N'):

In [12]:
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'N': 0, 'Y': 1})

In [13]:
df.head()

,person_age,person_income,person_emp_length,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,...,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,age_group_Middle-aged,age_group_Senior,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,59000,123.0,4,35000,16.02,1,0.59,1,3,...,False,False,True,False,False,False,False,False,True,False
1,21,9600,5.0,2,1000,11.14,0,0.10,0,2,...,False,True,False,False,False,True,False,False,False,False
2,25,9600,1.0,3,5500,12.87,1,0.57,0,3,...,False,False,False,False,False,False,False,True,False,False
3,23,65500,4.0,3,35000,15.23,1,0.53,0,2,...,False,False,True,False,False,False,False,True,False,False
4,24,54400,8.0,3,35000,14.27,1,0.55,1,4,...,False,False,True,False,False,False,False,True,False,False


## 6. Save Processed Data

After cleaning the data and performing feature engineering, the processed dataset is saved for future use in modeling. The following code is used to save the processed dataset as a CSV file:

In [14]:
df.to_csv('../data/clean_credit_risk_dataset.csv', index=False)

## Conclusion

Overall, this notebook has successfully performed feature engineering on the credit risk dataset. The data has been cleaned, new features have been created, and categorical variables have been encoded. The processed dataset is now ready for modeling, which will be in the next notebook in the credit risk project.